# 07 — Drift detection baseline

This notebook demonstrates Population Stability Index (PSI) on one numeric feature. Drift is a signal to investigate, not proof that model quality has degraded.

In [1]:
import json, os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

def psi(reference, current, bins=10):
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins+1)))
    if len(edges) < 3: return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    ref_pct = np.histogram(reference, bins=edges)[0] / len(reference)
    cur_pct = np.histogram(current, bins=edges)[0] / len(current)
    eps=1e-6
    return float(np.sum((cur_pct-ref_pct)*np.log((cur_pct+eps)/(ref_pct+eps))))
feature = X.columns[0]
reference = X_train[feature].to_numpy()
current = X_test[feature].to_numpy()
score = psi(reference, current)
print({'feature':feature, 'psi':round(score,4), 'alert':score >= 0.2})

{'feature': 'mean radius', 'psi': 0.0309, 'alert': False}


### Interpretation guide

- PSI below 0.1: often treated as small change
- 0.1–0.2: moderate change; inspect context
- 0.2 or higher: potentially meaningful change; investigate

These are example heuristics, not universal thresholds. Use domain-specific validation and monitor model quality when labels become available.